In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# QESEM: Una Qiskit Function de Qedma
*Consulta la [referencia de API](https://docs.quantum.ibm.com/api/functions/qedma-qesem)*

> **Note:** Las Qiskit Functions son una funcionalidad experimental disponible únicamente para usuarios de los planes IBM Quantum&reg; Premium, Flex y On-Prem (a través de la API de IBM Quantum Platform). Se encuentran en estado de versión preliminar y están sujetas a cambios.

## Descripción general
Si bien las unidades de procesamiento cuántico han mejorado enormemente en los últimos años, los errores debidos al ruido y las imperfecciones del hardware existente siguen siendo un desafío central para los desarrolladores de algoritmos cuánticos. A medida que el campo se acerca a computaciones cuánticas a escala utilitaria que no pueden verificarse de forma clásica, las soluciones para cancelar el ruido con precisión garantizada adquieren cada vez más importancia. Para superar este desafío, Qedma ha desarrollado Quantum Error Mitigation (QESEM), integrado de forma transparente en IBM Quantum Platform como una [Qiskit Function](/guides/functions).

Con QESEM, los usuarios pueden ejecutar sus circuitos cuánticos en QPUs ruidosas para obtener resultados altamente precisos y libres de errores con sobrecargas de tiempo de QPU muy eficientes, cercanas a los límites fundamentales. Para lograrlo, QESEM aprovecha un conjunto de métodos propietarios desarrollados por Qedma para la caracterización y reducción de errores. Las técnicas de reducción de errores incluyen optimización de puertas, transpilación consciente del ruido, supresión de errores (ES) y mitigación de errores imparcial (EM). Con esta combinación de métodos basados en caracterización, los usuarios pueden obtener resultados confiables y libres de errores para circuitos cuánticos genéricos de gran volumen, desbloqueando aplicaciones que de otro modo no podrían realizarse.

Para una descripción completa de los componentes subyacentes, así como una demostración a escala utilitaria, consulta el artículo [Reliable high-accuracy error mitigation for utility-scale quantum circuits](https://arxiv.org/abs/2508.10997).
## Descripción
Puedes usar la función QESEM de Qedma para estimar y ejecutar fácilmente tus circuitos con supresión y mitigación de errores, logrando mayores volúmenes de circuito y mayor precisión. Para usar QESEM, debes proporcionar un circuito cuántico, un conjunto de observables a medir, una precisión estadística objetivo para cada observable y una QPU seleccionada. Antes de ejecutar el circuito con la precisión objetivo, puedes estimar el tiempo de QPU necesario basándote en un cálculo analítico que no requiere ejecución del circuito. Una vez que estés satisfecho con la estimación de tiempo de QPU, puedes ejecutar el circuito con QESEM.

Cuando ejecutas un circuito, QESEM ejecuta un protocolo de caracterización del dispositivo adaptado a tu circuito, produciendo un modelo de ruido confiable para los errores que ocurren en él. A partir de la caracterización, QESEM primero implementa transpilación consciente del ruido para mapear el circuito de entrada a un conjunto de qubits físicos y puertas, minimizando el ruido que afecta al observable objetivo. Esto incluye las puertas disponibles de forma nativa (CX/CZ en dispositivos IBM&reg;), así como puertas adicionales optimizadas por QESEM, formando el conjunto de puertas extendido de QESEM. Luego, QESEM ejecuta un conjunto de circuitos ES y EM basados en caracterización en la QPU y recopila los resultados de sus mediciones. Estos se procesan clásicamente para proporcionar un valor de expectación imparcial y una barra de error para cada observable, correspondiente a la precisión solicitada.

![Descripción general de Qedma QESEM](../docs/images/guides/qedma-qesem/overview.svg)
Se ha demostrado que QESEM proporciona resultados de alta precisión para una variedad de aplicaciones cuánticas y en los mayores volúmenes de circuito alcanzables hoy en día. QESEM ofrece las siguientes características para el usuario, demostradas en la sección de benchmarks a continuación:
-	**Precisión garantizada:** QESEM genera estimaciones imparciales de los valores de expectación de los observables. Su método EM está equipado con garantías teóricas que, junto con la caracterización de vanguardia de Qedma, aseguran que la mitigación converge a la salida del circuito sin ruido hasta la precisión especificada por el usuario. A diferencia de muchos métodos EM heurísticos propensos a errores sistemáticos o sesgos, la precisión garantizada de QESEM es esencial para asegurar resultados confiables en circuitos cuánticos y observables genéricos.
-	**Escalabilidad a QPUs grandes:** El tiempo de QPU de QESEM depende del volumen del circuito, pero en lo demás es independiente del número de qubits. Qedma ha demostrado QESEM en los dispositivos cuánticos más grandes disponibles hoy, incluidos los dispositivos IBM Quantum Eagle de 127 qubits y Heron de 133 qubits.
-	**Agnóstico a la aplicación:** QESEM se ha demostrado en una variedad de aplicaciones, incluyendo simulación hamiltoniana, VQE, QAOA y estimación de amplitud. Los usuarios pueden ingresar cualquier circuito cuántico y observable a medir, y obtener resultados precisos y libres de errores. Las únicas limitaciones las dictan las especificaciones del hardware y el tiempo de QPU asignado, que determinan los volúmenes de circuito accesibles y las precisiones de salida. En contraste, muchas soluciones de reducción de errores son específicas de la aplicación o involucran heurísticas no controladas, lo que las hace inaplicables para circuitos y aplicaciones cuánticos genéricos.
-  **Conjunto de puertas extendido:** QESEM soporta puertas de ángulo fraccionario y proporciona puertas $Rzz(\theta)$ de ángulo fraccionario optimizadas por Qedma en dispositivos IBM Quantum Heron y Eagle. Este conjunto de puertas extendido permite una compilación más eficiente y desbloquea volúmenes de circuito mayores por un factor de hasta 2 en comparación con la compilación predeterminada de CX/CZ.
-	**Observables multibase:** QESEM soporta observables de entrada compuestos de muchas cadenas de Pauli no conmutantes, como Hamiltonianos genéricos. La elección de las bases de medición y la optimización de la asignación de recursos de QPU (shots y circuitos) se realiza automáticamente por QESEM para minimizar el tiempo de QPU requerido para la precisión solicitada. Esta optimización, que tiene en cuenta las fidelidades del hardware y las tasas de ejecución, te permite ejecutar circuitos más profundos y obtener mayor precisión.
## Benchmarks
QESEM ha sido probado en una amplia variedad de casos de uso y aplicaciones. Los siguientes ejemplos pueden ayudarte a evaluar qué tipos de cargas de trabajo puedes ejecutar con QESEM.

Una figura de mérito clave para cuantificar la dificultad tanto de la mitigación de errores como de la simulación clásica para un circuito y observable dado es el **volumen activo**: el número de puertas CNOT que afectan al observable en el circuito. El volumen activo depende de la profundidad y el ancho del circuito, del peso del observable y de la estructura del circuito, que determina el cono de luz del observable. Para más detalles, consulta la charla del [IBM Quantum Summit 2024](https://www.youtube.com/watch?v=Hd-IGvuARfE&t=1730s). QESEM proporciona un valor particularmente grande en el régimen de alto volumen, ofreciendo resultados confiables para circuitos y observables genéricos.

![Volumen activo](../docs/images/guides/qedma-qesem/active_volume.svg)

| Aplicación    | Número de qubits | Dispositivo | Descripción del circuito | Precisión | Tiempo total | Uso de runtime |
| ---------  | ---------------- | ----- | -------------------------- | -------- | ---------- | ------------- |
| Circuito VQE  | 8              | Eagle (r3) | 21 capas totales, 9 bases de medición, cadena 1D                    | 98%      | 35 min      | 14 min         |
| Kicked Ising   | 28              | Eagle (r3) | 3 capas únicas x 3 pasos, topología heavy-hex 2D                      | 97%     | 22 min      | 4 min          |
| Kicked Ising   | 28              | Eagle (r3) | 3 capas únicas x 8 pasos, topología heavy-hex 2D                     | 97%      | 116 min      | 23 min          |
| Simulación hamiltoniana Trotterizada   | 40  | Eagle (r3)            | 2 capas únicas x 10 pasos de Trotter, cadena 1D                    | 97%      | 3 horas     | 25 min         |
| Simulación hamiltoniana Trotterizada   | 119  | Eagle (r3)           | 3 capas únicas x 9 pasos de Trotter, topología heavy-hex 2D                    | 95%      | 6,5 horas     | 45 min         |
| Kicked Ising  | 136             | Heron (r2) | 3 capas únicas x 15 pasos, topología heavy-hex 2D                    | 99%      | 52 min             | 9 min           |

La precisión se mide aquí en relación con el valor ideal del observable: $\frac{\langle O \rangle_{ideal} - \epsilon}{\langle O \rangle_{ideal}}$, donde '$\epsilon$' es la precisión absoluta de la mitigación (establecida por el usuario), y $\langle O \rangle_{ideal}$ es el observable del circuito sin ruido.
El «uso de runtime» mide el uso del benchmark en modo batch (suma sobre el uso de los trabajos individuales), mientras que el «tiempo total» mide el uso en modo sesión (tiempo de pared del experimento), que incluye tiempos adicionales de comunicación y procesamiento clásico. QESEM está disponible para ejecución en ambos modos, de modo que los usuarios puedan aprovechar al máximo sus recursos disponibles.

Los circuitos Kicked Ising de 28 qubits simulan el Cristal de Tiempo Discreto estudiado por Shinjo et al. (ver [arXiv 2403.16718](https://arxiv.org/abs/2403.16718) y [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)) en tres bucles conectados de ibm_kawasaki. Los parámetros del circuito tomados aquí son $(\theta_x, \theta_z) = (0.9 \pi, 0)$, con un estado inicial ferromagnético $| \psi_0 \rangle = | 0 \rangle ^{\otimes n}$. El observable medido es el valor absoluto de la magnetización $M = |\frac{1}{28} \sum_{i=0}^{27} \langle Z_i \rangle|$. El experimento Kicked Ising a escala utilitaria se ejecutó en los 136 mejores qubits de ibm_fez; este benchmark particular se ejecutó al ángulo de Clifford $(\theta_x, \theta_z) = (\pi, 0)$, en el que el volumen activo crece lentamente con la profundidad del circuito, lo que, junto con las altas fidelidades del dispositivo, permite alta precisión en un tiempo de ejecución corto.

Los circuitos de simulación hamiltoniana Trotterizada corresponden a un modelo de Ising de campo transverso con ángulos fraccionarios: $(\theta_{zz}, \theta_x) = (\pi / 4, \pi /8)$ y $(\theta_{zz}, \theta_x) = (\pi / 6, \pi / 8)$ respectivamente (ver [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)). El circuito a escala utilitaria se ejecutó en los 119 mejores qubits de ibm_brisbane, mientras que el experimento de 40 qubits se ejecutó en la mejor cadena disponible. La precisión se reporta para la magnetización; también se obtuvieron resultados de alta precisión para observables de mayor peso.

El circuito VQE fue desarrollado junto con investigadores del Center for Quantum Technology and Applications del Deutsches Elektronen-Synchrotron (DESY). El observable objetivo aquí fue un Hamiltoniano compuesto por un gran número de cadenas de Pauli no conmutantes, destacando el rendimiento optimizado de QESEM para observables multibases. La mitigación se aplicó a un ansatz optimizado clásicamente; aunque estos resultados aún no están publicados, se obtendrán resultados de la misma calidad para diferentes circuitos con propiedades estructurales similares.
## Comenzar
Autentícate usando tu [clave de API de IBM Quantum Platform](http://quantum.cloud.ibm.com/), y selecciona la Qiskit Function QESEM de la siguiente manera. (Este fragmento asume que ya has [guardado tu cuenta](/guides/functions#install-qiskit-functions-catalog-client) en tu entorno local.)

In [1]:
import qiskit

from qiskit_ibm_catalog import QiskitFunctionsCatalog


catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [2]:
# load the function
qesem_function = catalog.load("qedma/qesem")

## Examples

To get started, try this basic example of estimating the required QPU time to run QESEM for a given `pub`:

In [3]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [4]:
circ = qiskit.QuantumCircuit(5)
circ.cx(0, 1)
circ.cx(2, 3)
circ.cx(1, 2)
circ.cx(3, 4)

avg_magnetization = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("Z", [q], 1 / 5) for q in range(5)], num_qubits=5
)
other_observable = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("ZZ", [0, 1], 1.0), ("XZ", [1, 4], 0.5)], num_qubits=5
)

time_estimation_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    options={
        "estimate_time_only": "analytical",
    },
    backend_name=backend_name,  # E.g. "ibm_fez"
)

time_estimate_result = time_estimation_job.result()

Puedes usar las APIs familiares de Qiskit Serverless para verificar el estado de tu carga de trabajo de Qiskit Function o recuperar los resultados:

In [5]:
sample_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    backend_name=backend_name,  # E.g. "ibm_fez"
)

El siguiente fragmento de código describe cómo recuperar la estimación del tiempo de QPU (cuando `estimate_time_only` está establecido):

In [6]:
# Print the ID so you can use it later, if necessary
print(sample_job.job_id)

print(sample_job.status())
result = sample_job.result()

9a87a23f-82f5-429e-91fb-cc8a9d107980
QUEUED


El siguiente fragmento de código demuestra cómo recuperar los resultados de mitigación (cuando `estimate_time_only` no está establecido) y las métricas de ejecución. Estos contienen datos esenciales que permiten una comprensión más profunda de cómo los diferentes parámetros impactan la ejecución de QESEM. También puede ser relevante al escribir un artículo basado en tu investigación.

In [7]:
print(
    f"The estimated QPU time for this PUB is: "
    f"\n{time_estimate_result[0].metadata}"
)

The estimated QPU time for this PUB is: 
{'time_estimation_sec': 1800, 'description': None, 'instance': 'crn:v1:bluemix:public:quantum-computing:us-east:a/6c63dae5281147f1a0449b36e0aaba3a:ae40ab55-8c55-4042-9204-71a6541d56ec::', 'transpilation_level': 'standard', 'parallel_execution': False, 'total_qpu_time': 0, 'empirical_estimation_mitigation_results': None, 'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 42.44837867887691, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 17.879877626895905, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: EXECUTING_QPU': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}}}


## Obtener mensajes de error
Si el estado de tu carga de trabajo es ERROR, usa `job.result()` para recuperar el mensaje de error de la siguiente manera:

In [9]:
results = result[0]
print(f"Mitigated expectation values: \n{results.data.evs}")
print(f"Mitigated error-bar: \n{results.data.stds}")
noisy_results = results.metadata["noisy_results"]
print(f"Noisy expectation values: \n{noisy_results.evs}")
print(f"Noisy error-bar: \n{noisy_results.stds}")
print(f"Total QPU time: \n {results.metadata['total_qpu_time']}")
print(
    f"Gates fidelity measured during the experiment: "
    f"\n {results.metadata['gate_fidelities']}"
)
print(
    f"Total shots / mitigation shots: \n "
    f"{results.metadata['total_shots']} / "
    f"{results.metadata['mitigation_shots']}"
)
print("Transpiled circuits:")
for i, circuit in enumerate(results.metadata["transpiled_circs"]):
    print(f"Circuit {i}:")
    print(f"  Circuit: \n {circuit['circuit']}")
    if "qubit_map" in circuit:
        print(f"  Qubit mapping: \n {circuit['qubit_map']}")
    print(f"  Measurement bases: \n {circuit['num_measurement_bases']}")

Mitigated expectation values: 
[1.00169764 1.00460812]
Mitigated error-bar: 
[0.00155021 0.0099558 ]
Noisy expectation values: 
[0.95717143 0.94271429]
Noisy error-bar: 
[0.00206982 0.00872689]
Total QPU time: 
 150.0
Gates fidelity measured during the experiment: 
 {'CZ': 0.9979051606662408, 'ID1Q': 0.9993865847216725}
Total shots / mitigation shots: 
 495600 / 220400
Transpiled circuits:
Circuit 0:
  Circuit: 
 OPENQASM 3.0;
include "stdgates.inc";
bit[145] c0;
qubit[145] q0;
rz(-pi) q0[143];
rz(0) q0[141];
rz(-pi) q0[140];
sx q0[143];
sx q0[141];
sx q0[140];
rz(-pi/2) q0[143];
rz(pi/2) q0[141];
rz(-pi/2) q0[140];
sx q0[143];
sx q0[141];
sx q0[140];
rz(pi/2) q0[143];
rz(-pi/2) q0[141];
rz(pi/2) q0[140];
barrier q0[140], q0[141], q0[142], q0[143], q0[144];
cz q0[144], q0[143];
cz q0[142], q0[141];
barrier q0[144], q0[143], q0[142], q0[141];
barrier q0[140], q0[141], q0[142], q0[143], q0[144];
rz(-pi) q0[144];
rz(-pi/2) q0[143];
rz(0) q0[142];
rz(-pi/2) q0[141];
sx q0[144];
sx q0[143];

## Fetch error messages

If your workload status is ERROR, use `job.result()` to fetch the error message to fetch the error message as follows:

In [14]:
# Get the result and truncate for readability
result = sample_job.result()
result_str = str(result)
max_length = 500  # Adjust this value as necessary

if len(result_str) > max_length:
    truncated = (
        result_str[:max_length]
        + f"... (truncated {len(result_str) - max_length} characters)"
    )
else:
    truncated = result_str

print(truncated)

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(2,), dtype=float64>), stds=np.ndarray(<shape=(2,), dtype=float64>), shape=(2,)), metadata={'gate_fidelities': {'CZ': 0.9979051606662408, 'ID1Q': 0.9993865847216725}, 'total_shots': 495600, 'mitigation_shots': 220400, 'transpiled_circs': [{'circuit': 'OPENQASM 3.0;\ninclude "stdgates.inc";\nbit[145] c0;\nqubit[145] q0;\nrz(-pi) q0[143];\nrz(0) q0[141];\nrz(-pi) q0[140];\nsx q0[143];\nsx q0[141];\nsx q0[140];\nrz(-pi/2) q0[143];\nrz(pi... (truncated 4174 characters)


## Obtener soporte

¡El equipo de soporte de Qedma está aquí para ayudarte! Si encuentras algún problema o tienes preguntas sobre el uso de la Qiskit Function QESEM, no dudes en comunicarte con nosotros. Nuestro personal de soporte, competente y amable, está listo para asistirte con cualquier inquietud técnica o consulta que puedas tener.

Puedes enviarnos un correo electrónico a support@qedma.com para recibir asistencia. Por favor, incluye la mayor cantidad de detalles posible sobre el problema que estás experimentando para ayudarnos a proporcionar una respuesta rápida y precisa. También puedes contactar a tu representante POC dedicado de Qedma por correo electrónico o teléfono.

Para ayudarnos a asistirte de manera más eficiente, por favor proporciona la siguiente información cuando te comuniques con nosotros:

- Una descripción detallada del problema
- El ID del trabajo
- Cualquier mensaje o código de error relevante

Estamos comprometidos a brindarte soporte rápido y efectivo para garantizar que tengas la mejor experiencia posible con nuestra Qiskit Function.

¡Siempre buscamos mejorar nuestro producto y damos la bienvenida a tus sugerencias! Si tienes ideas sobre cómo podemos mejorar nuestros servicios o características que te gustaría ver, envíanos tus pensamientos a support@qedma.com.

## Próximos pasos

> **Tip:** - [Solicitar acceso a Qedma QESEM](https://quantum.cloud.ibm.com/functions?id=qedma-qesem).
> - Visita la [referencia de API](https://docs.quantum.ibm.com/api/functions/qedma-qesem) para esta Qiskit Function.
> - Revisar [Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199](https://arxiv.org/pdf/2507.01199).
> - Revisar [Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997](https://arxiv.org/pdf/2508.10997).